# 02 · Centrality & backbone

Imports come from the `eu_trade_network` package; this notebook orchestrates and visualises only.

**Betweenness note.** Weighted betweenness uses `distance = 1 / value_kusd` (larger trade ⇒ shorter path). Unweighted betweenness is the pure-topology view — and is identically zero on this complete digraph.

In [ ]:
from __future__ import annotations

import pandas as pd

from eu_trade_network import config, data_loader, db, graph, metrics, viz

## Graph + centralities

Rebuild the edge list / digraph for `config.YEAR`, compute strengths and centralities, then refresh the DuckDB `nodes` table.

In [ ]:
edgelist = data_loader.build_edgelist()
G = graph.build_graph(edgelist)
summary = graph.graph_summary(G)
cent = metrics.compute_centralities(G)

node_meta = (
    pd.DataFrame([{"iso3": n, **attrs} for n, attrs in G.nodes(data=True)])
    .sort_values("iso3")
    .reset_index(drop=True)
)
nodes_db = node_meta.merge(cent, on="iso3", how="inner")

con = db.connect()
db.init_schema(con)
db.write_table(
    con,
    "nodes",
    nodes_db[
        [
            "iso3",
            "name",
            "grp",
            "out_strength",
            "in_strength",
            "degree",
            "betweenness",
            "betweenness_u",
            "eigenvector",
            "pagerank",
        ]
    ],
)

# Keep edges in sync if NB01 has not been run in this environment.
edges_db = edgelist.copy()
edges_db["year"] = config.YEAR
db.write_table(con, "edges", edges_db[["exporter_iso3", "importer_iso3", "value_kusd", "year"]])

print(summary)
print(f"Updated {len(nodes_db)} nodes → {config.DB_PATH}")
cent.sort_values("betweenness", ascending=False).head(10)

## SQL rankings (RQ1)

Hub ranking by **weighted** betweenness, then Austria's bilateral partners.

In [ ]:
hubs = db.read_sql(con, config.SQL_DIR / "queries" / "01_top_hubs_by_betweenness.sql")
hubs

In [ ]:
aut_partners = db.read_sql(con, config.SQL_DIR / "queries" / "03_austria_trade_partners.sql")
aut_partners

In [ ]:
aut = hubs.loc[hubs["iso3"] == "AUT"].iloc[0]
aut_pr_rank = int(
    hubs.sort_values("pagerank", ascending=False)
    .reset_index(drop=True)
    .assign(r=lambda d: d.index + 1)
    .loc[lambda d: d["iso3"] == "AUT", "r"]
    .iloc[0]
)
aut_str_rank = int(
    cent.sort_values("out_strength", ascending=False)
    .reset_index(drop=True)
    .assign(r=lambda d: d.index + 1)
    .loc[lambda d: d["iso3"] == "AUT", "r"]
    .iloc[0]
)
n_zero_b = int((hubs["betweenness"] == 0).sum())
n_pos_b = int((hubs["betweenness"] > 0).sum())
top_hub = hubs.iloc[0]

print(
    f"Germany ({top_hub['iso3']}) leads weighted betweenness "
    f"({float(top_hub['betweenness']):.3f}); only {n_pos_b} of {len(hubs)} "
    f"economies have positive betweenness on this complete digraph "
    f"(density = {summary['density']:.2f}). "
    f"Austria is among the {n_zero_b} economies with betweenness = 0 "
    f"(SQL rank {int(aut['rank'])} is an arbitrary tie-break among zeros): "
    f"partners already trade directly, so AUT is not a value-weighted bridge. "
    f"By export strength Austria is {aut_str_rank}/{len(hubs)}; by PageRank "
    f"{aut_pr_rank}/{len(hubs)} — a mid-tier EU economy whose largest partner "
    f"is Germany on both sides of the trade balance."
)

## Strength distribution

The network is dense (near-complete bilateral coverage). We plot the strength ranking and **do not** fit a power law.

In [ ]:
fig_strength = viz.plot_degree_distribution(cent)
fig_strength.show()

## Disparity-filter backbone

Statistically significant edges at `config.DISPARITY_ALPHA` (Serrano, Boguñá & Vespignani 2009). Also compute the rich-club coefficient on the undirected projection.

In [ ]:
backbone = metrics.disparity_filter(G, alpha=config.DISPARITY_ALPHA)
bb_edges = pd.DataFrame(
    [
        {"exporter_iso3": u, "importer_iso3": v, "value_kusd": float(d["weight"])}
        for u, v, d in backbone.edges(data=True)
    ]
)
rc = metrics.rich_club(G)

print(
    f"Backbone keeps {backbone.number_of_edges()} / {G.number_of_edges()} edges "
    f"at alpha={config.DISPARITY_ALPHA}"
)
rc.tail(10)

In [ ]:
# Offline lon/lat map (avoids Plotly topojson CDN required by scattergeo).
fig_backbone = viz.plot_lonlat_flows(
    bb_edges,
    node_meta,
    top_n=min(150, len(bb_edges)),
    title=(
        f"Disparity-filter backbone (α={config.DISPARITY_ALPHA}): "
        f"{backbone.number_of_edges()} significant edges"
    ),
)
out = viz.save_fig(fig_backbone, "02_backbone_map.png", headline=True)
print(f"Saved {out}")
fig_backbone.show()

con.close()